In [ ]:
from pyspark.sql import *
from pyspark.sql.functions import *

spark=(
    SparkSession
    .builder
    .appName("string and dates")
    .master("local[*]")
    .getOrCreate()
)

In [ ]:
emp_data=[
    ["001", "101", "John Doe", "30", "Male", "50000", "2015-01-01"],
    ["002", "101", "Jane Smith", "25", "Female", "45000", "2016-02-15"],
    ["003", "102", "Bob Brown", "35", "Male", "55000", "2014-05-01"],
    ["004", "102", "Alice Lee", "28", "Female", "48000", "2017-09-30"],
    ["005", "103", "Jack Chan", "40", "Male", "60000", "2013-04-01"],
    ["006", "103", "Jill Wong", "32", "Female", "52000", "2018-04-01"],
    ["007", "101", "James Johnson", "42", "Male", "70000", "2012-03-15"],
    ["008", "102", "Kate Kim", "29", "Female", "51000", "2019-10-01"],
    ["009", "103", "Tom Tan", "33", "Male", "58000", "2016-06-01"],
    ["010", "104", "Lisa Lee", "27", "Female", "47000", "2018-08-01"],
    ["011", "104", "David Park", "38", "Male", "65000", "2015-11-01"],
    ["012", "105", "Susan Chan", "31", "Female", "54000", "2017-02-15"],
    ["013", "107", "Brian Kim", "45", "Male", "75000", "2011-07-01"],
    ["014", "107", "Emily Lee", "26", "Female", "46000", "2019-01-01"],
    ["015", "106", "Michael Lee", "37", "Male", "63000", "2014-09-30"],
    ["016", "107", "Kelly Chen", "30", "Female", "49000", "2018-04-01"],
    ["017", "105", "George Wang", "34", "Male", "57000", "2016-03-15"],
    ["018", "104", "Nancy Liu", "29", None, "50000", "2017-06-01"],
    ["019", "103", "Steven Chen", "36", "Male", "62000", "2018-08-01"],
    ["020", "107", "Grace Kim", "32", "Female", "53000", "2018-11-01"]
]
emp_schema="emp_id string,dept_id string,name string,age string,gender string,salary string,hire_date string"

emp=spark.createDataFrame(data=emp_data,schema=emp_schema)
emp.show()
emp.printSchema()

In [ ]:
emp_gender_fixed=emp.withColumn("new_gender",when(col("gender")=="Male",'M')
                                .when(col("gender")=="Female",'F')
                                .otherwise(None))
emp_gender_fixed.show()


In [ ]:
#emp_gender_fixed_1=emp.withColumn("new_gender",expr("case when gender='Male' then 'M' when gender='Female' then 'F' else null END"))
'''
emp_gender_fixed_1 = emp.withColumn(
    "new_gender",
    expr("""
        CASE
            WHEN gender = 'Male' THEN 'M'
            WHEN gender = 'Female' THEN 'F'
            ELSE NULL
        END
    """)
)
emp_gender_fixed_1.show()
'''


In [ ]:
emp_name_fixed=emp.withColumn("new_name",regexp_replace("name","J","Z"))
# emp_name_fixed.show()

In [ ]:
emp_date_fix=emp_name_fixed.withColumn("hire_date",to_date(("hire_date"),"yyyy-mm-dd"))
# emp_date_fix.printSchema()


In [ ]:
emp_dated=emp_date_fix.withColumn("date_now",current_date()).withColumn("timestamp_now",current_timestamp())
emp_dated.show(truncate=False)

In [ ]:
emp_1=emp_dated.na.drop()
emp_1.show()

In [ ]:
emp_null_df=emp_gender_fixed.withColumn("new_gender",coalesce("new_gender",lit("o")))
emp_null_df.show()

In [ ]:
emp_final=emp_null_df.drop("gender").withColumnRenamed("new_gender","gender")
emp_final.show()

In [ ]:
# emp_final.write.format("csv").save("data/output/4/emp.csv")

In [ ]:
emp_fixed=emp_final.withColumn("date_string",date_format("hire_date","dd/mm/yyyy")).withColumn("date_year",date_format("hire_date","yyyy"))
emp_fixed.show()